In [ ]:
"""
实验一：单一平台仿真实验（仅内容电商平台A）
"""

import os
import re
import json
import random
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from google.colab import drive

# ================= 配置区 =================
MODEL_PATH = "Qwen/Qwen2.5-14B-Instruct"
drive.mount('/content/drive')
BASE_DIR = "/content/drive/MyDrive/eco-simp"
MEMORY_DIR = os.path.join(BASE_DIR, "exp1_memories_v2_1")
RESULTS_DIR = os.path.join(BASE_DIR, "exp1_results_v2_1")
MAX_MEMORY_LINES = 30
NUM_ROUNDS = 5
EXIT_LOSE_ROUNDS = 2   # 连续2轮毛利<安全线退出
COOLDOWN_ROUNDS = 2     # 退出后冷却2轮可回

# 全局随机种子，确保可复现
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# 策略梯度：佣金3档 × 补贴3档 = 9组
#COMMISSION_LEVELS = [0.06, 0.09, 0.11, 0.14]
COMMISSION_LEVELS = [0.10, 0.14]
SUBSIDY_LEVELS = [0.01, 0.03, 0.05]

os.makedirs(RESULTS_DIR, exist_ok=True)

# ================= 1. 模型加载 =================
print(f"加载模型: {MODEL_PATH}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, device_map="auto", torch_dtype=torch.float16, trust_remote_code=True
)
print("模型加载完成")


def generate_llm_response(prompt, max_tokens=400):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_tokens,
            do_sample=True, temperature=0.75, top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    resp = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return resp.strip()


# ================= 2. Agent 基类 =================
class BaseAgent:
    def __init__(self, agent_id, role_type, desc):
        self.agent_id = agent_id
        self.role_type = role_type
        self.description = desc
        self.memory_file = os.path.join(MEMORY_DIR, f"{agent_id}_memory.txt")

    def init_memory(self):
        with open(self.memory_file, "w", encoding="utf-8") as f:
            f.write(f"【角色】{self.role_type}｜{self.description}\n【历史】\n")

    def read_memory(self):
        if not os.path.exists(self.memory_file):
            return ""
        with open(self.memory_file, "r", encoding="utf-8") as f:
            lines = f.readlines()
        if len(lines) <= MAX_MEMORY_LINES + 1:
            return "".join(lines)
        return "".join(lines[:1] + lines[-MAX_MEMORY_LINES:])

    def append_memory(self, text):
        with open(self.memory_file, "a", encoding="utf-8") as f:
            f.write(text + "\n")

    def decide(self, context, task):
        memory = self.read_memory()
        prompt = self.build_prompt(memory, context, task)
        res = generate_llm_response(prompt)
        think_match = re.search(r"<thinking>(.*?)</thinking>", res, re.DOTALL | re.I)
        dec_match = re.search(r"<decision>(.*?)</decision>", res, re.DOTALL | re.I)
        thinking = think_match.group(1).strip() if think_match else ""
        decision = dec_match.group(1).strip() if dec_match else res.strip()
        self.append_memory(f"情境：{context[:80]}...｜决策：{decision}")
        return decision, thinking

    def build_prompt(self, memory, context, task):
        return ""


# ================= 3. 商家Agent =================
SELLER_CONFIGS = [
    # (id, type, cost, price_min, price_max, margin_safe, ad_max, character)
    ("Seller_L1", "低价走量", 30, 45, 55, 0.00, 3, "保守求生，倾向低价低广告，追求稳定出单"),
    ("Seller_L2", "低价走量", 30, 45, 55, 0.00, 3, "薄利多销，愿意在定价区间中段定价换取更多销量"),
    ("Seller_L3", "低价走量", 30, 45, 55, 0.00, 3, "轻微卷价，喜欢比同行低1-2元抢单"),
    ("Seller_L4", "低价走量", 30, 45, 55, 0.00, 3, "广告厌恶型，几乎不投广告，依靠自然流量"),
    ("Seller_L5", "低价走量", 30, 45, 55, 0.00, 3, "跟随市场，参考上轮结果调整策略"),
    ("Seller_L6", "低价走量", 30, 45, 55, 0.00, 3, "激进投放型，愿意在低价品上投满广告博曝光"),
    ("Seller_M1", "中端品质", 50, 130, 170, 0.05, 15, "均衡稳健，追求利润与销量的平衡点"),
    ("Seller_M2", "中端品质", 50, 130, 170, 0.05, 15, "品牌导向，倾向高定价维护品牌调性，广告适度"),
    ("Seller_M3", "中端品质", 50, 130, 170, 0.05, 15, "转化优先，愿意降价提升转化率，广告投入较高"),
    ("Seller_H1", "高端溢价", 100, 270, 330, 0.10, 45, "高端长期主义，坚持高定价，重品牌形象和复购"),
]


class SellerAgent(BaseAgent):
    def __init__(self, sid, seller_type, cost, price_min, price_max, margin_safe, ad_max, character):
        super().__init__(sid, seller_type, f"{seller_type}｜{character}")
        self.seller_type = seller_type
        self.unit_cost = cost
        self.price_min = price_min
        self.price_max = price_max
        self.margin_safe = margin_safe
        self.ad_max = ad_max
        self.character = character
        self.last_margin = 0.0
        self.last_sales = 0
        self.consecutive_lose = 0
        self.exited = False
        self.cooldown_left = 0

    def reset(self):
        self.last_margin = 0.0
        self.last_sales = 0
        self.consecutive_lose = 0
        self.exited = False
        self.cooldown_left = 0

    def build_prompt(self, memory, context, task):
        margin_pct = f"{self.margin_safe:.0%}" if self.margin_safe > 0 else "0%（可微利生存）"
        return f"""你是一个在内容电商平台A上经营的商家，需要根据平台政策和市场环境做出理性的经营决策。

【商家身份】
- 类型：{self.seller_type}
- 经营性格：{self.character}
- 单位生产成本：￥{self.unit_cost}
- 允许定价区间：￥{self.price_min} ~ ￥{self.price_max}（你必须在此区间内选择一个具体价格，不能超出）
- 广告投入上限：￥{self.ad_max}（0到{self.ad_max}之间的任意金额）
- 毛利安全线：{margin_pct}

【经营常识提醒】
- 你的利润 = 销售收入 - 生产成本 - 平台佣金 - 广告费 - 达人佣金
- 定价越高，单件利润越高但可能卖得更少；定价越低，薄利但可能多销
- 广告投入越高，曝光排名越靠前（前2名高曝光，3-4名中曝光，5-6名低曝光），但广告费无论是否卖出都要支付
- 佣金率越高，你被扣的佣金越多；补贴率越高，用户实际支付更低，理论上有利于销量
- 请根据你的性格特点和上轮经营结果，做出差异化的策略选择

【历史记忆】
{memory}

【当前环境】
{context}

【决策任务】
{task}

请先进行简要的策略分析（考虑成本结构、竞争环境、你的性格倾向），然后给出具体决策。
严格按以下格式输出，价格必须是{self.price_min}到{self.price_max}之间的数字，广告必须是0到{self.ad_max}之间的数字：
<thinking>你的策略分析（2-3句话）</thinking>
<decision>价格:X,广告:Y</decision>"""

    def decide_action(self, policy):
        comm = policy["commission"]
        sub = policy["subsidy"]
        # 计算上轮的盈亏情况描述
        if self.last_sales > 0:
            margin_desc = f"上轮毛利率{self.last_margin:.1%}（{'盈利' if self.last_margin > self.margin_safe else '低于安全线'}），销量{self.last_sales}件"
        else:
            margin_desc = "上轮无销量" if self.last_margin == 0 and self.last_sales == 0 else f"上轮销量{self.last_sales}件"

        ctx = (f"平台A当前政策：佣金率={comm:.0%}（每笔交易抽取{comm:.0%}），补贴率={sub:.0%}（用户实付价格打{1-sub:.0%}折）。\n"
               f"你的经营状况：{margin_desc}。连续低于安全线轮次：{self.consecutive_lose}（达到{EXIT_LOSE_ROUNDS}轮将被迫退出）。\n"
               f"同类竞品约{sum(1 for c in SELLER_CONFIGS if c[1]==self.seller_type)}家在售。")
        task = (f"请决定本轮的商品定价（必须在￥{self.price_min}~￥{self.price_max}之间）和广告投入金额（￥0~￥{self.ad_max}之间）。\n"
                f"注意：广告费无论是否卖出都会扣除。定价时请考虑佣金{comm:.0%}和生产成本￥{self.unit_cost}的影响。")
        dec, thinking = self.decide(ctx, task)

        # 解析价格（带容错）
        price = (self.price_min + self.price_max) / 2  # 默认中位数而非下限
        price_match = re.findall(r'价格[:：]?\s*(\d+\.?\d*)', dec)
        if price_match:
            val = float(price_match[0])
            price = max(self.price_min, min(self.price_max, val))
        else:
            nums = re.findall(r'\d+\.?\d*', dec)
            for n in nums:
                val = float(n)
                if self.price_min <= val <= self.price_max:
                    price = val
                    break

        # 解析广告（带容错）
        ad = 0.0
        ad_match = re.findall(r'广告[:：]?\s*(\d+\.?\d*)', dec)
        if ad_match:
            val = float(ad_match[0])
            ad = max(0, min(self.ad_max, val))
        else:
            nums = re.findall(r'\d+\.?\d*', dec)
            for n in nums:
                val = float(n)
                if 0 <= val <= self.ad_max and val != price:
                    ad = val
                    break

        return {"price": round(price, 1), "ad": round(ad, 1), "thinking": thinking}


# ================= 4. 达人Agent =================
# 影响力写死，确保可复现
INFLUENCER_CONFIGS = [
    ("Infu_T1", "种草型", 3, 0.06, "温和走心，重视商品品质与粉丝信任，偏好推荐中高端好物", "中高端"),
    ("Infu_T2", "种草型", 2, 0.06, "性价比种草达人，善于发现高性价比商品，均衡务实", "中端"),
    ("Infu_D1", "带货型", 4, 0.04, "冲量带货王，追求单场最大销量，偏好低价爆品走量", "低价"),
    ("Infu_D2", "带货型", 3, 0.04, "稳健带货，追求稳定的佣金收益，不冒进", "中端"),
    ("Infu_D3", "带货型", 5, 0.04, "头部达人，影响力大，偏好高客单价商品以获得更高佣金总额", "中高端"),
]


class InfluencerAgent(BaseAgent):
    def __init__(self, iid, flavor, inf_power, comm_rate, personality, preference):
        super().__init__(iid, "达人", f"{flavor}｜{personality}")
        self.flavor = flavor
        self.inf_power = inf_power
        self.comm_rate = comm_rate
        self.personality = personality
        self.preference = preference
        self.chosen_seller = None

    def reset(self):
        self.chosen_seller = None

    def build_prompt(self, memory, context, task):
        return f"""你是一个在内容电商平台A上活跃的带货达人，需要从平台上的商家中选择一个进行合作带货。

【达人身份】
- 达人类型：{self.flavor}
- 性格特点：{self.personality}
- 带货影响力：{self.inf_power}/5（影响力越高，能为商家带来越多用户关注）
- 佣金比例：{self.comm_rate:.0%}（你从带货销售额中抽取的比例）
- 选品偏好：偏好{self.preference}商品

【选择逻辑提醒】
- 你的佣金收入 = 你带来的销售额 × {self.comm_rate:.0%}
- 高价商品单件佣金高，但转化可能低；低价商品单件佣金低，但走量大
- 请根据你的选品偏好和性格来选择最适合合作的商家
- 广告投入高的商家通常更重视销售，可能配合度更好

【历史记忆】
{memory}

【当前环境】
{context}

【决策任务】
{task}

请先分析各商家特点，结合你的偏好做出选择。
<thinking>你的分析（2-3句话）</thinking>
<decision>你选择的商家ID（如Seller_M1）</decision>"""

    def choose_seller(self, active_sellers_info):
        ctx = f"平台A上当前活跃的商家信息：\n{active_sellers_info}"
        task = (f"请从上述商家中选择一个进行合作带货。你偏好{self.preference}类型的商品。\n"
                f"请直接输出一个商家ID。格式：<decision>商家ID</decision>")
        dec, thinking = self.decide(ctx, task)

        self.chosen_seller = None
        for sid in re.findall(r'Seller_[A-Z]\d+', dec):
            self.chosen_seller = sid
            break
        return self.chosen_seller, thinking


# ================= 5. 用户Agent =================
USER_CONFIGS = []
_low_personas = [
    "极度价格敏感，只买最便宜的商品",
    "折扣猎手，有补贴优惠才下单",
    "容易被广告吸引，看到高曝光商品就想买",
    "信任达人推荐，达人说好就买",
    "平台忠实用户，习惯性浏览购买",
    "有品质意识但预算紧张，在预算内选最好的",
    "冲动型消费者，看到低价好物就下单",
    "理性比价型，会对比所有低价商品再决定",
    "回头客，倾向于回购之前买过的商家",
    "新用户，好奇心强，愿意尝试不同商家",
    "促销敏感型，补贴力度大就多买",
    "口碑导向，看重其他用户评价和达人推荐",
    "习惯购买型，偏好固定的低价商家",
    "社交推荐驱动，受达人影响大",
    "随机浏览型，没有强偏好，看到合适就买",
]
for i in range(15):
    USER_CONFIGS.append((f"User_L{i+1}", "低价敏感", 100, _low_personas[i]))

_mid_personas = [
    "重视达人推荐，达人背书的中端商品优先",
    "品质优先，愿意为更好品质多付一些",
    "价格适中即可，不追求最低也不追求最贵",
    "理性消费者，几乎不看广告，自主判断",
    "品牌忠诚型，认准的商家会持续复购",
    "尝鲜型，喜欢试新品和新商家",
    "精打细算型，追求品质的同时关注性价比",
    "口碑驱动，看重达人和其他用户的推荐",
    "活动敏感，补贴活动期间倾向多买",
    "均衡考量型，综合考虑价格、品质、推荐",
]
for i in range(10):
    USER_CONFIGS.append((f"User_M{i+1}", "品质性价比", 200, _mid_personas[i]))

_high_personas = [
    "只买高端精品，价格不是首要考虑因素",
    "深度信任头部达人推荐的高端商品",
    "纯品质导向，只关注产品本身的品质",
    "品牌忠诚度极高，只买认可的高端商家",
    "自主决策型，不受广告和达人影响，独立判断",
]
for i in range(5):
    USER_CONFIGS.append((f"User_H{i+1}", "高端品质", 400, _high_personas[i]))


class UserAgent(BaseAgent):
    def __init__(self, uid, user_type, budget, persona):
        super().__init__(uid, "用户", f"{user_type}｜{persona}")
        self.user_type = user_type
        self.budget = budget
        self.persona = persona

    def build_prompt(self, memory, context, task):
        # 不在这里随机，由外部传入
        return ""

    def build_purchase_prompt(self, memory, context, task, influenced_by_inf, influenced_by_ad):
        """构建购买决策的完整prompt，influenced状态由外部统一控制"""
        # 根据用户类型给出选购指导
        if self.user_type == "低价敏感":
            type_guide = (
                "作为低价敏感用户，你优先选择最便宜的商品。"
                "中端和高端商品通常超出你的预算，除非补贴后价格在你预算内，否则不要购买。"
                "你的核心诉求是花最少的钱买到能用的商品。"
            )
        elif self.user_type == "品质性价比":
            type_guide = (
                "作为品质性价比用户，你追求品质与价格的平衡。"
                "你倾向于购买中端商品，低价商品品质可能不够，高端商品超出你的预算考量。"
                "你会在预算范围内选择品质最好的商品。"
            )
        else:  # 高端品质
            type_guide = (
                "作为高端品质用户，你追求最好的产品品质。"
                "你倾向于购买高端或中高端商品，低价商品基本不在你的考虑范围内。"
                "价格不是你的首要顾虑，品质和品牌才是。"
            )

        inf_desc = "本轮你看到了达人的推荐内容，达人推荐会影响你的判断" if influenced_by_inf else "本轮你没有关注达人推荐，你更依赖自己的判断"
        ad_desc = "本轮你注意到了广告展示，高曝光商品吸引了你的注意" if influenced_by_ad else "本轮你对广告不敏感，广告曝光不影响你的选择"

        return f"""你是一个在内容电商平台A上购物的用户，需要决定是否购买以及买哪个商家的商品。

【用户身份】
- 用户类型：{self.user_type}
- 性格特点：{self.persona}
- 本轮预算上限：￥{self.budget}（你绝对不能购买单价超过￥{self.budget}的商品！）

【选购指导】
{type_guide}

【本轮状态】
- {inf_desc}
- {ad_desc}

【重要约束】
- 你最多购买2件商品，也可以选择不买
- 每件商品的价格必须 ≤ 你的预算￥{self.budget}，超过预算的商品禁止购买
- 请注意商品标注的实际支付价格（已含补贴折扣）

【历史记忆】
{memory}

【当前环境】
{context}

【决策任务】
{task}

请先分析哪些商品在你的预算内且符合你的偏好，然后做出决定。
<thinking>你的分析（考虑预算、偏好、达人推荐、广告影响）</thinking>
<decision>商家ID1,商家ID2</decision> 或 <decision>不买</decision>"""

    def decide_purchase(self, items_info, influencer_reco_info, influenced_by_inf, influenced_by_ad):
        """在平台A的商品中选择购买（最多2件），也可以不买"""
        memory = self.read_memory()
        ctx = f"平台A商品列表（价格为补贴后你需要实际支付的价格）：\n{items_info}\n\n达人推荐信息：\n{influencer_reco_info}"
        task = (f"从商品中选择购买（最多2件，也可以不买）。你的预算是￥{self.budget}，不能买超过这个价格的商品。"
                f"请输出你要购买的商家ID，用逗号分隔。如果不买则输出'不买'。"
                f"格式：<decision>商家ID1,商家ID2</decision> 或 <decision>不买</decision>")
        prompt = self.build_purchase_prompt(memory, ctx, task, influenced_by_inf, influenced_by_ad)
        res = generate_llm_response(prompt)

        think_match = re.search(r"<thinking>(.*?)</thinking>", res, re.DOTALL | re.I)
        dec_match = re.search(r"<decision>(.*?)</decision>", res, re.DOTALL | re.I)
        thinking = think_match.group(1).strip() if think_match else ""
        decision = dec_match.group(1).strip() if dec_match else res.strip()
        self.append_memory(f"情境：{ctx[:80]}...｜决策：{decision}")

        purchased = []
        if "不买" in decision or "none" in decision.lower():
            return purchased, thinking

        sids = re.findall(r'Seller_[A-Z]\d+', decision)
        for sid in sids[:2]:
            purchased.append(sid)
        return purchased, thinking


# ================= 6. 广告曝光分配（内容电商平台A） =================
def assign_exposure_A(seller_ads):
    """
    按广告投入排序：前2名高曝光，3-4名中曝光，5-6名低曝光，其余无曝光
    仅对有广告投入的商家分配曝光等级
    """
    sorted_sellers = sorted(seller_ads.items(), key=lambda x: x[1], reverse=True)
    ad_sellers = [(s, a) for s, a in sorted_sellers if a > 0]
    exposure = {}
    for i, (sid, _) in enumerate(ad_sellers):
        if i < 2:
            exposure[sid] = "高曝光"
        elif i < 4:
            exposure[sid] = "中曝光"
        elif i < 6:
            exposure[sid] = "低曝光"
        else:
            exposure[sid] = "无"
    return exposure


# ================= 7. 构建商品信息字符串 =================
def build_items_info(active_sellers, seller_actions, exposure_map, influencer_reco_map, subsidy_rate):
    """构建给用户看的商品列表（展示补贴后价格）"""
    lines = []
    for s in active_sellers:
        sid = s.agent_id
        act = seller_actions[sid]
        exp = exposure_map.get(sid, "无")
        inf_tag = influencer_reco_map.get(sid, "")
        original_price = act["price"]
        actual_price = round(original_price * (1 - subsidy_rate), 1)
        line = f"  {sid}({s.seller_type}) 实付￥{actual_price}（原价￥{original_price}，补贴{subsidy_rate:.0%}）"
        if exp != "无":
            line += f" 【{exp}】"
        if inf_tag:
            line += f" {inf_tag}"
        lines.append(line)
    return "\n".join(lines) if lines else "暂无商品"


# ================= 8. 仿真主逻辑 =================
def run_single_experiment(commission, subsidy, exp_label):
    """运行单组实验"""
    print(f"\n{'='*60}")
    print(f"实验组 {exp_label}: 佣金={commission:.2f}, 补贴={subsidy:.2f}")
    print(f"{'='*60}")

    mem_dir = os.path.join(MEMORY_DIR, exp_label)
    os.makedirs(mem_dir, exist_ok=True)

    policy = {"commission": commission, "subsidy": subsidy}

    # 初始化agents
    sellers = []
    for cfg in SELLER_CONFIGS:
        s = SellerAgent(*cfg)
        s.memory_file = os.path.join(mem_dir, f"{s.agent_id}_memory.txt")
        s.init_memory()
        s.reset()
        sellers.append(s)

    influencers = []
    for cfg in INFLUENCER_CONFIGS:
        inf = InfluencerAgent(*cfg)
        inf.memory_file = os.path.join(mem_dir, f"{inf.agent_id}_memory.txt")
        inf.init_memory()
        inf.reset()
        influencers.append(inf)

    users = []
    for cfg in USER_CONFIGS:
        u = UserAgent(*cfg)
        u.memory_file = os.path.join(mem_dir, f"{u.agent_id}_memory.txt")
        u.init_memory()
        users.append(u)

    # 数据记录
    round_records = []
    seller_records = []
    influencer_records = []
    user_records = []
    transaction_records = []

    for rnd in range(1, NUM_ROUNDS + 1):
        print(f"\n--- {exp_label} 第{rnd}轮 ---")

        # ---- 商家冷却处理 ----
        for s in sellers:
            if s.exited and s.cooldown_left > 0:
                s.cooldown_left -= 1
                if s.cooldown_left == 0:
                    s.exited = False
                    s.consecutive_lose = 0
                    print(f"  {s.agent_id} 冷却结束，重新回到平台A")

        # ---- 商家决策 ----
        active_sellers = [s for s in sellers if not s.exited]
        seller_actions = {}
        for s in active_sellers:
            act = s.decide_action(policy)
            seller_actions[s.agent_id] = act
            print(f"  {s.agent_id}: 价格={act['price']}, 广告={act['ad']}")

        # ---- 广告曝光分配 ----
        seller_ads = {sid: act["ad"] for sid, act in seller_actions.items()}
        exposure_map = assign_exposure_A(seller_ads)

        # ---- 达人选择商家 ----
        active_info = "\n".join([
            f"  {s.agent_id}({s.seller_type}) 定价￥{seller_actions[s.agent_id]['price']} 广告投入￥{seller_actions[s.agent_id]['ad']}"
            for s in active_sellers
        ])
        influencer_reco_map = {}
        influencer_chosen_sellers = {}  # {inf_id: seller_id} 记录每个达人选了谁
        influencer_round_data = []
        for inf in influencers:
            chosen_sid, inf_thinking = inf.choose_seller(active_info)
            if chosen_sid and chosen_sid in seller_actions:
                tag = f"【达人推荐:{inf.agent_id}(影响力{inf.inf_power}/5)】"
                if chosen_sid in influencer_reco_map:
                    influencer_reco_map[chosen_sid] += " " + tag
                else:
                    influencer_reco_map[chosen_sid] = tag
                influencer_chosen_sellers[inf.agent_id] = chosen_sid
            influencer_round_data.append({
                "exp": exp_label, "round": rnd,
                "influencer_id": inf.agent_id, "flavor": inf.flavor,
                "inf_power": inf.inf_power, "comm_rate": inf.comm_rate,
                "chosen_platform": "A", "chosen_seller": chosen_sid or "无",
                "thinking": inf_thinking
            })

        # ---- 构建商品展示（展示补贴后价格） ----
        items_info = build_items_info(active_sellers, seller_actions, exposure_map, influencer_reco_map, subsidy)
        reco_info = "\n".join([f"  {sid}: {tag}" for sid, tag in influencer_reco_map.items()]) if influencer_reco_map else "  无达人推荐"

        # ---- 用户购买 ----
        round_transactions = []
        user_round_data = []

        # 构建达人推荐的商家集合（用于判断订单是否归属达人）
        inf_recommended_sellers = set(influencer_chosen_sellers.values())
        # {seller_id: [inf_id, ...]} 反向映射
        seller_to_influencers = {}
        for inf_id, sid in influencer_chosen_sellers.items():
            seller_to_influencers.setdefault(sid, []).append(inf_id)

        for u in users:
            influenced_by_inf = random.choice([0, 1])
            influenced_by_ad = random.choice([0, 1])

            purchased_sids, u_thinking = u.decide_purchase(
                items_info, reco_info, influenced_by_inf, influenced_by_ad
            )

            valid_purchased = []
            for sid in purchased_sids:
                if sid in seller_actions:
                    actual_price = round(seller_actions[sid]["price"] * (1 - subsidy), 1)
                    if actual_price <= u.budget:
                        valid_purchased.append(sid)
                    else:
                        print(f"    ⛔ {u.agent_id} 试图购买 {sid}(￥{actual_price})超出预算￥{u.budget}，已拦截")

            if not valid_purchased:
                user_round_data.append({
                    "exp": exp_label, "round": rnd,
                    "user_id": u.agent_id, "user_type": u.user_type,
                    "persona": u.persona, "budget": u.budget,
                    "influenced_by_influencer": influenced_by_inf,
                    "influenced_by_ad": influenced_by_ad,
                    "platform": "A", "purchased": 0,
                    "seller_bought": "", "price_paid": 0,
                    "thinking": u_thinking
                })
            else:
                for sid in valid_purchased:
                    price = seller_actions[sid]["price"]
                    actual_price = round(price * (1 - subsidy), 1)
                    # 判断该订单是否受达人影响（用于达人佣金归属）
                    is_influencer_driven = (influenced_by_inf == 1 and sid in inf_recommended_sellers)
                    round_transactions.append({
                        "exp": exp_label, "round": rnd,
                        "user_id": u.agent_id, "user_type": u.user_type,
                        "seller_id": sid, "platform": "A",
                        "original_price": price, "subsidy_rate": subsidy,
                        "actual_price": actual_price,
                        "is_influencer_driven": is_influencer_driven
                    })
                    user_round_data.append({
                        "exp": exp_label, "round": rnd,
                        "user_id": u.agent_id, "user_type": u.user_type,
                        "persona": u.persona, "budget": u.budget,
                        "influenced_by_influencer": influenced_by_inf,
                        "influenced_by_ad": influenced_by_ad,
                        "platform": "A", "purchased": 1,
                        "seller_bought": sid, "price_paid": actual_price,
                        "is_influencer_driven": is_influencer_driven,
                        "thinking": u_thinking
                    })

        # ---- 清算 ----
        seller_sales = {s.agent_id: 0 for s in sellers}
        seller_gmv = {s.agent_id: 0.0 for s in sellers}
        seller_inf_driven_gmv = {s.agent_id: 0.0 for s in sellers}  # 达人带来的GMV

        for tx in round_transactions:
            seller_sales[tx["seller_id"]] += 1
            seller_gmv[tx["seller_id"]] += tx["original_price"]
            if tx["is_influencer_driven"]:
                seller_inf_driven_gmv[tx["seller_id"]] += tx["original_price"]

        # 计算达人佣金（仅基于达人带来的GMV）
        seller_influencer_cost = {s.agent_id: 0.0 for s in sellers}
        for inf in influencers:
            if inf.chosen_seller and inf.chosen_seller in seller_inf_driven_gmv:
                # 如果多个达人推荐同一商家，均分该商家的达人驱动GMV
                num_infs = len(seller_to_influencers.get(inf.chosen_seller, []))
                if num_infs > 0:
                    share_gmv = seller_inf_driven_gmv[inf.chosen_seller] / num_infs
                    seller_influencer_cost[inf.chosen_seller] += share_gmv * inf.comm_rate

        for s in active_sellers:
            sid = s.agent_id
            act = seller_actions[sid]
            sales = seller_sales[sid]
            gmv = seller_gmv[sid]

            comm_cost = gmv * commission             # 平台佣金
            ad_cost = act["ad"]                      
            total_cost = s.unit_cost * sales          # 生产成本
            inf_cost = seller_influencer_cost[sid]    # 达人佣金
            profit = gmv - total_cost - comm_cost - ad_cost - inf_cost
            margin = profit / gmv if gmv > 0 else (-ad_cost / 1 if ad_cost > 0 else 0)

            s.last_margin = margin
            s.last_sales = sales

            # 退出判断
            if gmv == 0 and ad_cost > 0:
                # 花了广告费但没卖出，算亏损
                s.consecutive_lose += 1
            elif gmv > 0 and margin < s.margin_safe:
                s.consecutive_lose += 1
            elif gmv > 0:
                s.consecutive_lose = 0
            # gmv==0 且 ad_cost==0 的情况不累计

            if s.consecutive_lose >= EXIT_LOSE_ROUNDS and not s.exited:
                s.exited = True
                s.cooldown_left = COOLDOWN_ROUNDS
                print(f"  ⚠ {sid} 连续{EXIT_LOSE_ROUNDS}轮亏损，退出平台A，冷却{COOLDOWN_ROUNDS}轮")

            seller_records.append({
                "exp": exp_label, "round": rnd,
                "seller_id": sid, "seller_type": s.seller_type,
                "character": s.character, "unit_cost": s.unit_cost,
                "price": act["price"], "ad_amount": act["ad"],
                "exposure": exposure_map.get(sid, "无"),
                "sales": sales, "gmv": gmv,
                "commission_cost": round(comm_cost, 2),
                "production_cost": round(total_cost, 2),
                "influencer_cost": round(inf_cost, 2),
                "influencer_driven_gmv": round(seller_inf_driven_gmv[sid], 2),
                "ad_cost": round(ad_cost, 2),
                "profit": round(profit, 2),
                "margin": round(margin, 4),
                "margin_safe": s.margin_safe,
                "consecutive_lose": s.consecutive_lose,
                "exited": s.exited,
                "cooldown_left": s.cooldown_left,
                "thinking": act["thinking"]
            })

        # 退出的商家也要记录
        for s in sellers:
            if s.exited and s.agent_id not in seller_actions:
                seller_records.append({
                    "exp": exp_label, "round": rnd,
                    "seller_id": s.agent_id, "seller_type": s.seller_type,
                    "character": s.character, "unit_cost": s.unit_cost,
                    "price": 0, "ad_amount": 0, "exposure": "无",
                    "sales": 0, "gmv": 0,
                    "commission_cost": 0, "production_cost": 0,
                    "influencer_cost": 0, "influencer_driven_gmv": 0,
                    "ad_cost": 0, "profit": 0, "margin": 0,
                    "margin_safe": s.margin_safe,
                    "consecutive_lose": s.consecutive_lose,
                    "exited": True,
                    "cooldown_left": s.cooldown_left,
                    "thinking": "已退出平台"
                })

        # ---- 汇总本轮 ----
        total_gmv = sum(seller_gmv.values())
        total_trans = len(round_transactions)
        active_count = len(active_sellers)
        exited_count = sum(1 for s in sellers if s.exited)

        platform_comm_income = sum(seller_gmv[s.agent_id] * commission for s in active_sellers)
        platform_sub_expense = sum(seller_gmv[s.agent_id] * subsidy for s in active_sellers)
        platform_ad_income = sum(seller_actions[s.agent_id]["ad"] for s in active_sellers)  # 所有广告费都是平台收入
        platform_revenue = platform_comm_income - platform_sub_expense + platform_ad_income

        round_records.append({
            "exp": exp_label, "round": rnd,
            "commission": commission, "subsidy": subsidy,
            "total_gmv": round(total_gmv, 2),
            "total_transactions": total_trans,
            "active_sellers": active_count,
            "exited_sellers": exited_count,
            "platform_comm_income": round(platform_comm_income, 2),
            "platform_sub_expense": round(platform_sub_expense, 2),
            "platform_ad_income": round(platform_ad_income, 2),
            "platform_revenue": round(platform_revenue, 2),
            "influencers_on_A": len(influencers),
            "users_purchased": sum(1 for u_data in user_round_data if u_data.get("purchased") == 1),
            "influencer_driven_transactions": sum(1 for tx in round_transactions if tx["is_influencer_driven"])
        })

        influencer_records.extend(influencer_round_data)
        user_records.extend(user_round_data)
        transaction_records.extend(round_transactions)

        print(f"  GMV={total_gmv:.0f} | 交易={total_trans} | 达人带货={sum(1 for tx in round_transactions if tx['is_influencer_driven'])} | 活跃商家={active_count} | 退出={exited_count} | 平台营收={platform_revenue:.0f}")

    return round_records, seller_records, influencer_records, user_records, transaction_records


# ================= 9. 运行所有9组实验 =================
all_round = []
all_seller = []
all_influencer = []
all_user = []
all_transaction = []

exp_idx = 0
for comm in COMMISSION_LEVELS:
    for sub in SUBSIDY_LEVELS:
        exp_idx += 1
        label = f"C{comm:.2f}_S{sub:.2f}"
        rr, sr, ir, ur, tr = run_single_experiment(comm, sub, label)
        all_round.extend(rr)
        all_seller.extend(sr)
        all_influencer.extend(ir)
        all_user.extend(ur)
        all_transaction.extend(tr)

        # 每组实验后保存
        pd.DataFrame(all_round).to_csv(f"{RESULTS_DIR}/exp1_round_summary.csv", index=False, encoding="utf-8-sig")
        pd.DataFrame(all_seller).to_csv(f"{RESULTS_DIR}/exp1_seller_detail.csv", index=False, encoding="utf-8-sig")
        pd.DataFrame(all_influencer).to_csv(f"{RESULTS_DIR}/exp1_influencer_detail.csv", index=False, encoding="utf-8-sig")
        pd.DataFrame(all_user).to_csv(f"{RESULTS_DIR}/exp1_user_detail.csv", index=False, encoding="utf-8-sig")
        pd.DataFrame(all_transaction).to_csv(f"{RESULTS_DIR}/exp1_transactions.csv", index=False, encoding="utf-8-sig")

        print(f"\n✅ 实验组 {label} ({exp_idx}/9) 完成，数据已保存")

print(f"\n{'='*60}")
print(f"✅ 实验一全部完成！共{exp_idx}组实验，数据保存在 {RESULTS_DIR}")
print(f"{'='*60}")

Mounted at /content/drive
加载模型: Qwen/Qwen2.5-14B-Instruct


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

... (3 outputs left out)

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

模型加载完成

实验组 C0.10_S0.01: 佣金=0.10, 补贴=0.01

--- C0.10_S0.01 第1轮 ---
  Seller_L1: 价格=45, 广告=1.0
  Seller_L2: 价格=47.0, 广告=2.0
  Seller_L3: 价格=46.0, 广告=2.0
  Seller_L4: 价格=47.0, 广告=0.5
  Seller_L5: 价格=47.0, 广告=2.0
  Seller_L6: 价格=48.0, 广告=3
  Seller_M1: 价格=145.0, 广告=8.0
  Seller_M2: 价格=150.0, 广告=8.0
  Seller_M3: 价格=150.0, 广告=10.0
  Seller_H1: 价格=300.0, 广告=20.0
  GMV=4913 | 交易=49 | 达人带货=20 | 活跃商家=10 | 退出=0 | 平台营收=499

--- C0.10_S0.01 第2轮 ---
  Seller_L1: 价格=45, 广告=0
  Seller_L2: 价格=47.0, 广告=2.0
  Seller_L3: 价格=45, 广告=2.5
  Seller_L4: 价格=45, 广告=0
  Seller_L5: 价格=46.0, 广告=3
  Seller_L6: 价格=47.0, 广告=3
  Seller_M1: 价格=145.0, 广告=10.0
  Seller_M2: 价格=145.0, 广告=6.0
  Seller_M3: 价格=140.0, 广告=10.0
  Seller_H1: 价格=300.0, 广告=30.0
  ⚠ Seller_L6 连续2轮亏损，退出平台A，冷却2轮
  GMV=4967 | 交易=50 | 达人带货=7 | 活跃商家=10 | 退出=1 | 平台营收=514

--- C0.10_S0.01 第3轮 ---
  Seller_L1: 价格=45, 广告=0
  Seller_L2: 价格=47.0, 广告=2.0
  Seller_L3: 价格=45, 广告=2.5
  Seller_L4: 价格=45, 广告=0
  Seller_L5: 价格=48.0, 广告=2.0
  Seller_M1: 价格=145.0, 广告=10